# Track 4b realism check: the channel-faithful cohort P and the membership ablation (Colab)

Pre-registered in `specs/track4_priors_probes/gate_report.md` → **R.1–R.4**, before this run.
**Generated by `build/mk_np_signals.py`; do not hand-edit.** Core: `build/np_signals_core.py`.

This reuses the first run's outputs in `casmi_2026/_np_signals/` (`train_labels.parquet`, `lotus_fp.npy`,
`lotus_superclass.npy`), so there is no SID-Map pass. Runtime: **CPU, High-RAM**, about 1–2 h (benchmark only).

| cohort | what | window |
|---|---|---|
| P | every NP-table compound **missing from the pool** (150–1250 Da) | same-formula tier isomers **minus pool members**: what supernext's PubChem channel can list |
| L, T | same seed and size as the first run | full same-formula window (for the ablation) |

Variants: `pop`, `nplike@μ`, `classcons@μ`, **`member@μ`** (has an NP label), **`stack@μ`** (both terms).
Paste the final printout back into the chat.

In [ ]:
import os, sys, json, time, subprocess
import multiprocessing as mp
from pathlib import Path
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
TIER_DRIVE = ROOT / "casmi26-pubchem-tier"
POP_DRIVE = ROOT / "casmi26-pubchem-popularity-prior"
NPT_DRIVE = ROOT / "casmi26-natural-product-knowledge-table"
NPS = ROOT / "_np_signals"                     # outputs of casmi_np_signals_colab.ipynb
WORK = Path("/content/np_realism")
DL = Path("/content/np_signals/dl")             # same local staging folder as the first notebook
for p in (WORK, DL):
    p.mkdir(parents=True, exist_ok=True)
PY_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
RDKIT_WHL = next(iter(sorted((ROOT / "casmi26-offline-rdkit-2026033").glob(f"rdkit-*-{PY_TAG}-*.whl"))), None)
RDKIT_PIN = "2026.03.3"

# pre-registered (gate_report.md R.*): cohort L/T re-use the first run's seed and size
N_COHORT, SEED, PPM_WIN = 2000, 0, 2.0
R1_MIN = 0.005

NP_CORE_SRC = '"""Track 4b NP signals: the core functions shared by the Colab build/benchmark and the local tests.\n\nEmbedded verbatim into casmi_np_signals_colab.ipynb (by build/mk_np_signals.py) and written back to a file,\nso the multiprocessing workers can import it. Plan: specs/track4_priors_probes/np_signals_plan.md.\n\nThree signals, each meant as one additive term on top of the borrowed popularity prior\npop = log1p(SID) + log1p(PMID):\n  vendor_only  1 if every live SID of the CID was deposited by a "Chemical Vendors" source and PMID == 0\n  np_like      Ertl NP-likeness (RDKit Contrib NP_Score), z-scored within each candidate window\n  class_cons   superclass agreement between a candidate and the molecule\'s nearest library structures,\n               similarity-weighted. A candidate with no label scores 0, never a penalty.\n"""\nfrom __future__ import annotations\n\nimport os\nimport re\nimport sys\n\nimport numpy as np\n\n# ---------------------------------------------------------------- metrics\n\ndef expected_rank_metrics(scores, truth_idx, k=25, hits=(1, 5, 25)):\n    """Expected reciprocal rank @k and hit@n of scores[truth_idx] under uniformly random tie-breaking.\n\n    Ties matter here: thousands of vendor decoys share pop = log1p(1) and, under a pessimistic or optimistic\n    tie rule, the ordering of equal scores would decide the metric on its own.\n    Returns (rr, {n: hit}).\n    """\n    s = np.round(np.asarray(scores, np.float64), 9)\n    t = s[truth_idx]\n    g = int((s > t).sum())\n    e = int((s == t).sum())               # includes the truth itself\n    ranks = np.arange(g + 1, g + e + 1)\n    rr = float(np.where(ranks <= k, 1.0 / ranks, 0.0).mean())\n    hit = {n: float(np.clip(n - g, 0, e) / e) for n in hits}\n    return rr, hit\n\n\n# ---------------------------------------------------------------- vendor_only\n\ndef vendor_source_ids(source_table, source_names=None):\n    """Return the set of source names/ids whose PubChem source category contains "vendor".\n\n    source_table: DataFrame from PUG REST sourcetable/substance/CSV; must have a name column and a category\n    column (case-insensitive match on \'categ\'). Raises if no category column exists, because a silent fallback\n    would make the vendor flag mean something else.\n    source_names: optional DataFrame (id, display_name) from FTP Substance/Extras/Source-Names. SID-Map\n    stores the depositor\'s source *id*, which for newer depositors is numeric, so ids are added for every\n    vendor display name.\n    """\n    cols = {c.lower(): c for c in source_table.columns}\n    cat = next((cols[c] for c in cols if \'categ\' in c), None)\n    if cat is None:\n        raise KeyError(f\'no category column in the source table: {list(source_table.columns)}\')\n    name = next((cols[c] for c in cols if c in (\'source name\', \'name\', \'sourcename\', \'source\')), None)\n    if name is None:\n        name = next(cols[c] for c in cols if \'name\' in c)\n    isv = source_table[cat].astype(str).str.contains(\'vendor\', case=False, na=False)\n    vend = set(source_table.loc[isv, name].astype(str).str.strip())\n    if source_names is not None:\n        sid_col, disp_col = source_names.columns[:2]\n        m = source_names[disp_col].astype(str).str.strip().isin(vend) | source_names[sid_col].astype(str).str.strip().isin(vend)\n        vend |= set(source_names.loc[m, sid_col].astype(str).str.strip())\n    return vend\n\n\ndef accumulate_sid_map(src, cid, vendor_set, nonvendor, anysid):\n    """One SID-Map chunk: src (array of source ids, str), cid (float/int array, NaN = no standardized CID).\n    Sets anysid[cid] for every SID and nonvendor[cid] for SIDs whose source is not a vendor. In place."""\n    cid = np.asarray(cid, np.float64)\n    ok = np.isfinite(cid) & (cid > 0) & (cid < len(nonvendor))\n    if not ok.any():\n        return 0\n    import pandas as pd\n    c = cid[ok].astype(np.int64)\n    isv = pd.Series(np.asarray(src, object)[ok]).astype(str).str.strip().isin(vendor_set).values\n    anysid[c] = True\n    nonvendor[c[~isv]] = True\n    return int(ok.sum())\n\n\ndef vendor_only_flags(cids, lpmid, nonvendor, anysid):\n    """Per-row vendor_only for an array of representative CIDs and log1p(PMID) values."""\n    cids = np.asarray(cids, np.int64)\n    ok = (cids > 0) & (cids < len(nonvendor))\n    c = np.where(ok, cids, 0)\n    no_pmid = np.asarray(lpmid, np.float32) <= 0\n    return (ok & anysid[c] & ~nonvendor[c] & no_pmid).astype(np.uint8)\n\n\n# ---------------------------------------------------------------- NP-likeness\n\n_NP = {}\n\n\ndef _np_model():\n    if \'m\' not in _NP:\n        from rdkit.Chem import RDConfig\n        p = os.path.join(RDConfig.RDContribDir, \'NP_Score\')\n        if p not in sys.path:\n            sys.path.append(p)\n        import npscorer\n        _NP[\'f\'] = npscorer\n        _NP[\'m\'] = npscorer.readNPModel()\n    return _NP[\'f\'], _NP[\'m\']\n\n\ndef np_scores(smiles):\n    """Ertl NP-likeness per SMILES; NaN where RDKit cannot parse."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    f, m = _np_model()\n    out = np.full(len(smiles), np.nan, np.float32)\n    for i, s in enumerate(smiles):\n        mol = Chem.MolFromSmiles(s)\n        if mol is not None:\n            try:\n                out[i] = f.scoreMol(mol, m)\n            except Exception:\n                pass\n    return out\n\n\ndef zscore_window(x):\n    """z-score within one candidate window; NaN -> 0 (neutral), constant window -> all 0."""\n    x = np.asarray(x, np.float64)\n    ok = np.isfinite(x)\n    out = np.zeros(len(x))\n    if ok.sum() >= 2:\n        sd = x[ok].std()\n        if sd > 0:\n            out[ok] = (x[ok] - x[ok].mean()) / sd\n    return out\n\n\n# ---------------------------------------------------------------- class consistency\n\ndef morgan_fp(smiles, nbits=2048):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog(\'rdApp.*\')\n    gen = _NP.setdefault(\'gen\', rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=nbits))\n    m = Chem.MolFromSmiles(smiles)\n    return None if m is None else gen.GetFingerprint(m)\n\n\ndef knn_label(fp, bank_fps, bank_labels, thr=0.7):\n    """Label of the nearest bank structure if its Tanimoto >= thr, else None. Returns (label, sim)."""\n    if fp is None or not len(bank_fps):\n        return None, 0.0\n    from rdkit import DataStructs\n    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, bank_fps))\n    j = int(sims.argmax())\n    return (bank_labels[j], float(sims[j])) if sims[j] >= thr else (None, float(sims[j]))\n\n\ndef nearest_analogs(fp, bank_fps, k=5, exclude=None):\n    """Top-k bank indices by Tanimoto (excluding indices in `exclude`). Returns (idx, sims)."""\n    from rdkit import DataStructs\n    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, bank_fps))\n    if exclude is not None and len(exclude):\n        sims[np.asarray(list(exclude), np.int64)] = -1.0\n    idx = np.argsort(-sims)[:k]\n    idx = idx[sims[idx] > 0]\n    return idx, sims[idx]\n\n\ndef class_consistency(cand_labels, analog_labels, analog_sims):\n    """s_c = sum_k sim_k [label_c == label_k] / sum_k sim_k over labelled analogs. Unlabelled candidate -> 0;\n    no labelled analog -> all 0 (the signal abstains instead of guessing)."""\n    pairs = [(l, w) for l, w in zip(analog_labels, analog_sims) if l is not None and w > 0]\n    out = np.zeros(len(cand_labels))\n    if not pairs:\n        return out\n    tot = sum(w for _, w in pairs)\n    wt = {}\n    for l, w in pairs:\n        wt[l] = wt.get(l, 0.0) + w / tot\n    for i, l in enumerate(cand_labels):\n        if l is not None:\n            out[i] = wt.get(l, 0.0)\n    return out\n\n\n# ---------------------------------------------------------------- PubChem-tier windows\n\n_FORMULA_RE = re.compile(r\'([A-Z][a-z]?)(\\d*)\')\n\n\ndef canon_formula(f):\n    """Hill-order-insensitive formula key (\'C6H12O6\' == \'H12C6O6\'); charges dropped."""\n    f = re.sub(r\'[+-]\\d*$\', \'\', str(f))\n    d = {}\n    for el, n in _FORMULA_RE.findall(f):\n        d[el] = d.get(el, 0) + (int(n) if n else 1)\n    return \'\'.join(f\'{el}{d[el]}\' for el in sorted(d))\n\n\ndef tier_slice(mass, off, buf, target, ppm):\n    """Row range [a, b) and SMILES of the tier rows within ppm of a neutral mass."""\n    tol = target * ppm * 1e-6\n    a = int(np.searchsorted(mass, target - tol, \'left\'))\n    b = int(np.searchsorted(mass, target + tol, \'right\'))\n    if b <= a:\n        return a, b, []\n    o = np.asarray(off[a:b + 1], np.int64)\n    raw = bytes(buf[o[0]:o[-1]])\n    base = o[0]\n    return a, b, [raw[o[i] - base:o[i + 1] - base].decode(\'ascii\') for i in range(b - a)]\n\n\ndef same_formula(smiles, formula):\n    """Mask of SMILES whose molecular formula equals `formula` (canonicalised)."""\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    RDLogger.DisableLog(\'rdApp.*\')\n    want = canon_formula(formula)\n    ok = np.zeros(len(smiles), bool)\n    for i, s in enumerate(smiles):\n        m = Chem.MolFromSmiles(s)\n        if m is not None and canon_formula(CalcMolFormula(m)) == want:\n            ok[i] = True\n    return ok\n\n\ndef dedupe_first(keys):\n    """Indices of the first occurrence of each key, in order."""\n    seen, keep = set(), []\n    for i, k in enumerate(keys):\n        if k not in seen:\n            seen.add(k)\n            keep.append(i)\n    return np.asarray(keep, np.int64)\n\n\n# ---------------------------------------------------------------- offline benchmark (one query)\n\nG = {}                 # filled by the notebook before the worker Pool forks (mmaps, banks, label maps)\nMUS = (0.25, 0.5, 1.0)\nKNN_TOP = 150          # candidates (by pop) that get a kNN label when the NP table has none\nKNN_THR = 0.7\nK_ANALOG = 5\n\n\ndef _key(k):\n    return k.decode(\'ascii\', \'ignore\') if isinstance(k, (bytes, np.bytes_)) else str(k)\n\n\ndef eval_query(task):\n    """task = (qid, ik14, formula, mono_mass, smiles, cohort). Ranks the truth among its same-formula PubChem-tier\n    isomers (deduplicated by IK14, keeping the most popular row) under pop and pop + mu * signal.\n\n    G keys: mass, off, buf (tier), ik14, lsid, lpmid, vendor (row-aligned), ppm, label_of (ik14 -> superclass),\n    lotus_fps, lotus_labels, train_fps, train_labels, train_rows_of (ik14 -> list of bank indices).\n    """\n    qid, ik14, formula, mono_mass, smiles, cohort = task\n    out = dict(qid=qid, cohort=cohort, found=False, n=0)\n    a, b, smis = tier_slice(G[\'mass\'], G[\'off\'], G[\'buf\'], float(mono_mass), G[\'ppm\'])\n    if not smis:\n        return out\n    sf = np.where(same_formula(smis, formula))[0]\n    if not len(sf):\n        return out\n    rows = a + sf\n    pop = np.asarray(G[\'lsid\'][rows], np.float32) + np.asarray(G[\'lpmid\'][rows], np.float32)\n    keys = np.array([_key(k) for k in G[\'ik14\'][rows]], object)\n    o = np.argsort(-pop, kind=\'stable\')\n    keep = o[dedupe_first(keys[o])]\n    keep = keep[keys[keep] != \'\']\n    rows, pop, keys = rows[keep], pop[keep], keys[keep]\n    excl = G.get(\'exclude_keys\') if cohort in G.get(\'exclude_cohorts\', ()) else None\n    if excl:\n        # mirror supernext\'s PubChem channel: structures already in the candidate pool are never listed there\n        m = np.array([(k == ik14) or (k not in excl) for k in keys], bool)\n        rows, pop, keys = rows[m], pop[m], keys[m]\n    t = np.where(keys == ik14)[0]\n    out[\'n\'] = int(len(rows))\n    if not len(t):\n        return out\n    t = int(t[0])\n    out[\'found\'] = True\n    sm = [smis[i - a] for i in rows]\n    vendor = np.asarray(G[\'vendor\'][rows], np.float64)\n    npz = zscore_window(np_scores(sm))\n\n    # class consistency: candidate labels (table, else kNN for the KNN_TOP most popular), analogs from the library\n    lab = [G[\'label_of\'].get(k) for k in keys]\n    for i in np.argsort(-pop, kind=\'stable\')[:KNN_TOP]:\n        if lab[i] is None:\n            lab[i], _ = knn_label(morgan_fp(sm[i]), G[\'lotus_fps\'], G[\'lotus_labels\'], KNN_THR)\n    qfp = morgan_fp(smiles)\n    if qfp is not None and len(G[\'train_fps\']):\n        ai, asim = nearest_analogs(qfp, G[\'train_fps\'], K_ANALOG, exclude=G[\'train_rows_of\'].get(ik14, ()))\n        alab = [G[\'train_labels\'][j] for j in ai]\n    else:\n        alab, asim = [], []\n    cc = class_consistency(lab, alab, asim)\n    mem = np.array([l is not None for l in lab], np.float64)   # membership-only ablation: "has an NP label"\n    top_lab = {lab[i] for i in np.argsort(-pop, kind=\'stable\')[:KNN_TOP] if lab[i] is not None}\n    out.update(has_lab_analog=any(l is not None for l in alab), multi_class_window=len(top_lab) >= 2,\n               truth_vendor=int(vendor[t]), truth_lab=lab[t] is not None, cc_truth=float(cc[t]),\n               vendor_rate=float(vendor.mean()))\n\n    variants = {\'pop\': pop}\n    for mu in MUS:\n        variants[f\'vendor@{mu}\'] = pop - mu * vendor\n        variants[f\'nplike@{mu}\'] = pop + mu * npz\n        variants[f\'classcons@{mu}\'] = pop + mu * cc\n        variants[f\'member@{mu}\'] = pop + mu * mem\n        variants[f\'stack@{mu}\'] = pop + mu * npz + mu * cc\n    for name, sc in variants.items():\n        rr, hit = expected_rank_metrics(sc, t)\n        out[f\'rr|{name}\'] = rr\n        for n, h in hit.items():\n            out[f\'hit{n}|{name}\'] = h\n    return out\n'
(WORK / "np_signals_core.py").write_text(NP_CORE_SRC)
sys.path.insert(0, str(WORK))

In [ ]:
print("python", PY_TAG, "| offline wheel:", RDKIT_WHL.name if RDKIT_WHL else "none for this Python -> PyPI")
cmd = ([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(RDKIT_WHL)] if RDKIT_WHL is not None
       else [sys.executable, "-m", "pip", "install", "-q", f"rdkit=={RDKIT_PIN}"])
r_ = subprocess.run(cmd, capture_output=True, text=True)
if r_.returncode != 0:
    raise RuntimeError("rdkit install failed:\n" + r_.stdout[-2000:] + "\n" + r_.stderr[-2000:])
import rdkit
assert rdkit.__version__ == RDKIT_PIN, rdkit.__version__
import np_signals_core as C
print("rdkit", rdkit.__version__, "| cpus", os.cpu_count())

In [ ]:
T0 = time.time()

def find(dst, name):
    hits = sorted(dst.rglob(name), key=lambda p: len(str(p)))
    assert hits, f"{name} not found under {dst}"
    return hits[0]

def stage(src_dir, names, dst):
    dst.mkdir(parents=True, exist_ok=True)
    for n in names:
        s = find(src_dir, n); d = dst / n
        if not (d.exists() and d.stat().st_size == s.stat().st_size):
            t_ = time.time(); subprocess.run(["cp", str(s), str(d)], check=True)
            print(f"  copied {n} {s.stat().st_size/1e9:.2f} GB in {time.time()-t_:.0f}s", flush=True)
    return dst

for d in (TIER_DRIVE, POP_DRIVE, NPT_DRIVE, NPS):
    assert d.exists(), f"missing on Drive: {d}"
TIER = stage(TIER_DRIVE, ["pc_mass.npy", "pc_off.npy", "pc_smiles.npy"], DL / "tier")
POP = stage(POP_DRIVE, ["pc_ik14.npy", "pc_lsid.npy", "pc_lpmid.npy"], DL / "pop")
mass = np.load(TIER / "pc_mass.npy", mmap_mode="r")
off = np.load(TIER / "pc_off.npy", mmap_mode="r")
buf = np.load(TIER / "pc_smiles.npy", mmap_mode="r")
pc_ik14 = np.load(POP / "pc_ik14.npy", mmap_mode="r")
pc_lsid = np.load(POP / "pc_lsid.npy", mmap_mode="r")
pc_lpmid = np.load(POP / "pc_lpmid.npy", mmap_mode="r")
pc_vendor = np.load(NPS / "pc_vendor_only.npy", mmap_mode="r")
assert len(mass) == len(pc_ik14) == len(pc_vendor) == 105_875_489
npt = pd.read_parquet(find(NPT_DRIVE, "natural_products.parquet"))
tr = pd.read_parquet(NPS / "train_labels.parquet")
lotus_fp = np.load(NPS / "lotus_fp.npy")
lotus_lab = np.load(NPS / "lotus_superclass.npy", allow_pickle=True)

# pool keys (train U COCONUT): pool_meta.parquet `key` if casmi26-v2-pool is on Drive, else the NP table's
# in_casmi_pool rows plus the training structures (the popularity dataset has no pool key array)
PM = [p for p in [ROOT / "casmi26-v2-pool" / "pool_meta.parquet", *ROOT.glob("*/pool_meta.parquet"),
                  *ROOT.glob("*/*/pool_meta.parquet")] if p.exists()]
if PM:
    POOL_KEYS = set(pd.read_parquet(PM[0], columns=["key"]).key.astype(str)); POOL_SRC = str(PM[0])
else:
    pr = npt[npt.in_casmi_pool.astype(bool)]
    POOL_KEYS = set(pr.ik14.astype(str)) | set(pr.scorer_key.astype(str)) | set(tr.ik.astype(str)); POOL_SRC = "NP table + train"
POOL_KEYS -= {"", "nan", "None"}
print("pool keys from", POOL_SRC)
print(f"inputs ready {time.time()-T0:.0f}s | pool keys {len(POOL_KEYS)} | train {len(tr)} | LOTUS bank {len(lotus_fp)}")

In [ ]:
from rdkit import DataStructs
T1 = time.time()
lab = npt[npt.np_superclass.notna()].drop_duplicates("ik14")
LABEL_OF = dict(zip(lab.ik14, lab.np_superclass))
LOTUS_FPS = [DataStructs.CreateFromBitString("".join(map(str, np.unpackbits(b)))) for b in lotus_fp]

# the first run's cohorts, rebuilt with the same RNG calls in the same order
rng = np.random.default_rng(SEED)
cT = tr[(tr.mono >= 150) & (tr.mono <= 1250)]
cT = cT.iloc[rng.permutation(len(cT))[:N_COHORT]]
cL = npt[(~npt.in_casmi_train.astype(bool)) & npt.mono_mass.between(150, 1250)].drop_duplicates("ik14")
cL = cL.iloc[rng.permutation(len(cL))[:N_COHORT]]
cP = npt[(~npt.in_casmi_pool.astype(bool)) & npt.mono_mass.between(150, 1250)].drop_duplicates("ik14")
cP = cP[~cP.ik14.isin(POOL_KEYS)]
tasks = [(f"T{i}", r.ik, r.formula, r.mono, r.normalized_smiles, "T") for i, r in enumerate(cT.itertuples())]
tasks += [(f"L{i}", r.ik14, r.formula, r.mono_mass, r.smiles, "L") for i, r in enumerate(cL.itertuples())]
tasks += [(f"P{i}", r.ik14, r.formula, r.mono_mass, r.smiles, "P") for i, r in enumerate(cP.itertuples())]
print("cohorts: T", len(cT), "| L", len(cL), "| P", len(cP))

C.G.update(mass=mass, off=off, buf=buf, ik14=pc_ik14, lsid=pc_lsid, lpmid=pc_lpmid, vendor=pc_vendor, ppm=PPM_WIN,
           label_of={**LABEL_OF, **{k: v for k, v in zip(tr.ik, tr.superclass) if v is not None}},
           lotus_fps=LOTUS_FPS, lotus_labels=list(lotus_lab),
           train_fps=[C.morgan_fp(s) for s in tr.normalized_smiles], train_labels=list(tr.superclass),
           train_rows_of={k: [i] for i, k in enumerate(tr.ik)},
           exclude_keys=POOL_KEYS, exclude_cohorts=("P",))
rows = []
with mp.get_context("fork").Pool(os.cpu_count()) as P:
    for k, r in enumerate(P.imap_unordered(C.eval_query, tasks, chunksize=4)):
        rows.append(r)
        if (k + 1) % 250 == 0:
            print(f"  {k+1}/{len(tasks)} {time.time()-T1:.0f}s", flush=True)
B = pd.DataFrame(rows)
B.to_parquet(NPS / "np_realism_rows.parquet", index=False)

In [ ]:
print("\n=== Track 4b realism check ===")
F = B[B.found].copy()
for co in ("T", "L", "P"):
    b = B[B.cohort == co]
    print(f"cohort {co}: {len(b)} sampled, found {b.found.mean():.1%}, median window {int(b[b.found].n.median())}")
names = sorted({c.split("|", 1)[1] for c in F.columns if c.startswith("rr|")}, key=lambda s: (s != "pop", s))
tab = []
for nm in names:
    row = dict(variant=nm)
    for co in ("T", "L", "P"):
        f = F[F.cohort == co]
        row[f"MRR_{co}"] = f[f"rr|{nm}"].mean()
    tab.append(row)
TAB = pd.DataFrame(tab)
for co in ("T", "L", "P"):
    TAB[f"d_{co}"] = TAB[f"MRR_{co}"] - TAB.loc[TAB.variant == "pop", f"MRR_{co}"].iloc[0]
pd.set_option("display.width", 200)
print(TAB.round(4).to_string(index=False))

def d(v, co):
    return float(TAB.loc[TAB.variant == v, f"d_{co}"].iloc[0])

R = {}
R["R1"] = dict(value=d("classcons@0.5", "L") - d("member@0.5", "L"), thr=R1_MIN)
R["R1"]["pass_"] = R["R1"]["value"] >= R1_MIN
R["R2"] = dict(value=d("classcons@0.5", "P"), thr=0.0); R["R2"]["pass_"] = R["R2"]["value"] >= 0
R["R3"] = dict(value=d("nplike@0.5", "P"), thr=0.0); R["R3"]["pass_"] = R["R3"]["value"] >= 0
best_single = max(d("classcons@0.5", "P"), d("nplike@0.5", "P"))
R["R4"] = dict(value=d("stack@0.5", "P") - best_single, thr=0.0); R["R4"]["pass_"] = R["R4"]["value"] > 0
fp_ = F[F.cohort == "P"]
R["P_truth_labelled"] = float(fp_.truth_lab.mean()) if len(fp_) else float("nan")
R["P_found"] = float(B[B.cohort == "P"].found.mean())
for k in ("R1", "R2", "R3", "R4"):
    v = R[k]
    print(f"{k}: {v['value']:+.4f} (thr {v['thr']:+.3f}) -> {'PASS' if v['pass_'] else 'FAIL'}")
print(f"cohort P: truth labelled {R['P_truth_labelled']:.1%} | found {R['P_found']:.1%}")
R["table"] = TAB.round(5).to_dict(orient="records")
(NPS / "np_realism.json").write_text(json.dumps(R, indent=1, default=float))
print(f"wrote {NPS / 'np_realism.json'} | total {time.time()-T0:.0f}s")